## Initialization

### Imports

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log, ceil
import shutil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import make_interp_spline
from pint import Quantity
import bottleneck

from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
from data_processing import helpers
# from data_processing.paths import (
#     get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    # NonReactorDataframeColumn,
    # SliceFitDataframeColumn,
    EnergyColumn,
    get_df_col
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing.processing.neutron_window_strategy.strategy_factory \
    import NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy \
    import AbstractNeutronStrategy
# from data_processing.helpers import (
#     # get_input_with_default,
#     # get_input_required,
#     # input_experiment_ids,
#     stop,
#     get_midpoints_from_bins
# )


### Functions

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> proc_types.NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = proc_types.NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


In [ ]:
# def relative_rmse(x: pd.Series | float, x_err: pd.Series | float, y: pd.Series | float, y_err: pd.Series | float) -> pd.Series | float:
def relative_rmse(values: list[tuple[pd.Series | float, pd.Series | float]]) -> pd.Series | float:
    rel_sq_values = [relative_square_error(x, x_err) for x, x_err in values]
    # rel_sq_x = relative_square_error(x, x_err)
    # rel_sq_y = relative_square_error(y, y_err)
    # rel_sq_sum = rel_sq_x + rel_sq_y
    rel_sq_sum = sum(rel_sq_values)
    if isinstance(rel_sq_sum, pd.Series):
        return rel_sq_sum.pow(1./2)
    else:
        return rel_sq_sum ** (1./2)


def relative_square_error(x: pd.Series | float, x_err: pd.Series | float) -> pd.Series | float:
    # divide x_err by x
    # square it
    # return
    rel_err = x_err / x
    if isinstance(rel_err, pd.Series):
        return rel_err.pow(2).fillna(0)
    else:
        return rel_err ** 2

In [ ]:
def correct_raw_signals(
    raw_signals_df: pd.DataFrame,
    baseline_idx_range: int = 40,
    baseline_offset: float = 0,
    max_adc: int = 16367,
    use_max_adc: bool = False
) -> pd.DataFrame:
    offset = int(baseline_offset * max_adc)
    signals_np = raw_signals_df.to_numpy()
    
    if use_max_adc:
        baselines = max_adc
    else:
        baselines = signals_np[
            :, :baseline_idx_range
        ].mean(axis=1).reshape(-1, 1)
    
    signals_np = -signals_np + baselines + offset
    corrected_signals = pd.DataFrame(
        signals_np,
        index=raw_signals_df.index,
        columns=raw_signals_df.columns
    )
    return corrected_signals

In [ ]:
def get_psd_adc_histogram(
    df: pd.DataFrame,
    adc_col: str = DetectorDataframeColumn.ENERGY.value,
    psd_col: str = DetectorDataframeColumn.PSD.value,
    adc_width: float = 420,
    adc_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = df[adc_col]
    y = df[psd_col]

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    if adc_bins is not None:
        x_bins = adc_bins
    else:
        x_bins: np.ndarray = np.linspace(
            0, x.max(), int(x.max() / adc_width) + 1
        )
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

In [ ]:
def integrate_pulses(pulse_data: np.array, t_start: int, t_end: int):
    left_vals = pulse_data[:, t_start:t_end]
    right_vals = pulse_data[:, t_start+1:t_end+1]
    # left_vals = pulse_series.loc[t_start:t_end]
    # right_vals = pulse_series.loc[t_start+1:t_end+1]

    # print(left_vals.shape, right_vals.shape)
    # print(left_vals)
    # print(right_vals)
    midpoints = (left_vals + right_vals) / 2
    # print(midpoints)
    column_areas = midpoints * 2  # 2 ns between data points
    # print(column_areas)
    areas = column_areas.sum(axis=1)
    return areas


def integrate_pulse_gates(
    pulses: pd.DataFrame, t1: int, t2: int, t3: int, baseline_count: int = 40
) -> pd.DataFrame:
    baseline_adjust = 0
    # baseline_window = 25

    if not pd.api.types.is_numeric_dtype(pulses.values):
        raise ValueError("DataFrame values must all be numeric type")
    if pulses.shape[1] != 200:
        raise ValueError("DataFrame rows must be 200 samples long")
    if not all([0 <= x <= 398 for x in [t1, t2, t3]]):
        raise ValueError("Times must all be between 0 and 398 (inclusive)")
    if not (t2 > t1):
        raise ValueError("t2 must be greater than t1")
    if not (t3 > t2):
        raise ValueError("t3 must be greater than t2")

    idx_1 = ceil(t1 / 2)
    idx_2 = ceil(t2 / 2)
    idx_3 = ceil(t3 / 2)

    pulses_np = pulses.to_numpy()
    # baselines = np.trunc(
    #     pulses_np[:, :baseline_count].mean(axis=1).reshape(-1, 1)
    # )
    # baselines = baselines + baseline_adjust
    # long_slices = pulses_np[:, idx_1:idx_3]
    # short_slices = pulses_np[:, idx_1:idx_2]
    # peak_slices = pulses_np[:, 30:60]

    # q_long = (baselines - long_slices).sum(axis=1)
    # q_short = (baselines - short_slices).sum(axis=1)
    q_long = integrate_pulses(pulses_np, idx_1, idx_3)
    q_short = integrate_pulses(pulses_np, idx_1, idx_2)
    # peak_heights = (baselines - peak_slices).max(axis=1)

    psd_df = pd.DataFrame(
        {"Q_LONG": q_long, "Q_SHORT": q_short},
        index=pulses.index
    )
    return psd_df

In [ ]:
def two_point_inv_lerp(y: float, p1: tuple[float, float], p2: tuple[float, float]) -> float:
    deltas = tuple([n2 - n1 for n1, n2 in zip(p1, p2)])
    m = deltas[1] / deltas[0]
    x1, y1 = p1
    if m == 0:
        return x1
    x = (y - y1) / m + x1
    return x


def calculate_q_fom_critical(
    fit_df: pd.DataFrame,
    # q_limits: tuple[float, float] | None = None
    # q_limit_hi: float | None = None
) -> tuple[float | None, float | None]:
    fom_crit = 1.27
    # if q_limits is None:
    #     q_limits = (2500, 60000)  # x axis area with clean FOM curve
    # if q_limit_hi is None:
    #     q_limit_hi = 60000
    fom_data = fit_df["fom"]
    slice_energy_min = fit_df["slice_energy_min"]
    slice_energy_max = fit_df["slice_energy_max"]
    slice_energy_mid = (slice_energy_min + slice_energy_max) / 2
    fom_x = slice_energy_mid.values
    fom_y = fom_data.values
    delta_y = fom_y[2:] - fom_y[:-2]
    stable_end_idx = np.argmax(fom_x >= 20000)

    # use moving average of delta_y to find stable region (delta_y <= threshold)
    # find first cross in stable region
    window = 5
    # bottleneck window functions use look-behind windows and fill missing with nan
    # so first window-1 values are always nan; we need to convert to look-ahead
    # 
    delta_y_mov_max = bottleneck.move_max(
        delta_y[:stable_end_idx+window-1], window
    )[window-1:]
    delta_y_mov_min = bottleneck.move_min(
        delta_y[:stable_end_idx+window-1], window
    )[window-1:]
    stable_max_delta = delta_y_mov_max < 0.25
    stable_min_delta = delta_y_mov_min > -0.25
    stable_delta = (stable_max_delta & stable_min_delta)
    if not stable_delta.any():
        return None, None
    stable_start_idx = np.argmax(stable_delta)
    stable_start_x = fom_x[stable_start_idx]
    cross_search_slice = fom_y[stable_start_idx:stable_end_idx]
    if not (cross_search_slice >= fom_crit).any():
        return None, stable_start_x
    # argmax gets i in slice, need to add slice start to get i in original list
    fom_critical_idx = np.argmax(cross_search_slice >= fom_crit) + stable_start_idx
    
    # possible_crosses = np.where((fom_y[1:] >= 1.27) & (fom_y[:-1] <= 1.27))[0]
    # # dd_sums = []
    # fom_critical_idx = None
    # for possible_cross in possible_crosses:
    #     pass  # STUB
    # slice_lo = possible_cross - 3 if possible_cross - 3 >= 0 else 0
    # slice_hi = possible_cross + 2
    # deltas = delta_y[slice_lo:slice_hi]
    # delta_deltas = deltas[1:] - deltas[:-1]
    # dd_sum = abs(delta_deltas).sum()
    # dd_sums.append(dd_sum)
    
    # idx_best_cross = np.argmin(np.nan_to_num(dd_sums, nan=np.inf))
    # fom_critical_idx = possible_crosses[idx_best_cross] + 1
    if fom_critical_idx is None:
        q_fom_critical = None
    elif fom_critical_idx > 0:
        # x_crit_bounds = tuple([fom_x[fom_critical_idx+x] for x in [-1, 0]])
        # y_crit_bounds = tuple([fom_y[fom_critical_idx+x] for x in [-1, 0]])
        p1 = fom_x[fom_critical_idx-1], fom_y[fom_critical_idx-1]
        p2 = fom_x[fom_critical_idx], fom_y[fom_critical_idx]
        if p1[1] > fom_crit:  # we can't make lerp extrapolate!
            q_fom_critical = None
        else:
            q_fom_critical = two_point_inv_lerp(fom_crit, p1, p2)
    else:
        q_fom_critical = fom_x[fom_critical_idx]
    return q_fom_critical, stable_start_x

In [ ]:
def calculate_q_fom_critical_from_pulses(
    times: np.ndarray, pulses: pd.DataFrame
) -> tuple[float | None, float | None]:
    t1, t2, t3, *_ = times.flatten()
    psd_df = integrate_pulses(pulses, t1, t2, t3)
    psd_df = calculate_psd(psd_df)
    histogram, energy_bin_edges, psd_bin_edges = get_psd_energy_histogram(psd_df)
    fit_df, _ = scan_histogram_slices(histogram, energy_bin_edges, psd_bin_edges)
    q_fom_critical = calculate_q_fom_critical(fit_df)
    return q_fom_critical


def generate_search_grid(
    t1_limits: tuple[float, float],
    t2_limits: tuple[float, float],
    t3_limits: tuple[float, float],
    counts: int | tuple[int, int, int],
) -> np.ndarray:
    if isinstance(counts, tuple):
        t1_counts, t2_counts, t3_counts = counts
    else:
        t1_counts, t2_counts, t3_counts = counts, counts, counts
    t1_range = np.linspace(*t1_limits, num=t1_counts)
    t2_range = np.linspace(*t2_limits, num=t2_counts)
    t3_range = np.linspace(*t3_limits, num=t3_counts)
    t_grid = np.meshgrid(t1_range, t2_range, t3_range)
    t_grid = tuple([np.ravel(grid_element) for grid_element in t_grid])
    t_grid_stacked = np.vstack(t_grid)  # shape (3, n)
    return t_grid_stacked


T = TypeVar("T")


def search_grid(
    grid_search_fn: Callable[[np.ndarray, pd.DataFrame], T],
    t1_limits: tuple[float, float],
    t2_limits: tuple[float, float],
    t3_limits: tuple[float, float],
    counts: int | tuple[int, int, int],
    pulses: pd.DataFrame,
    cores: int = 4,
    use_chunks: bool = False,
) -> tuple[np.ndarray, list[T]]:
    t_grid_stacked = generate_search_grid(
        t1_limits, t2_limits, t3_limits, counts
    )

    pool_size = max(
        2 * cores, 4
    )
    # based on https://jupyter-tutorial.readthedocs.io/en/stable
    # /performance/multiprocessing.html
    if use_chunks:
        chunksize, extra = divmod(len(t_grid_stacked.shape[1]), pool_size * 4)
        if extra > 0:
            chunksize += 1
    else:
        chunksize = "auto"

    parallelizer = Parallel(
        n_jobs=pool_size, batch_size=chunksize, max_nbytes=1e6, verbose=10
    )
    loop_result = parallelizer(
        delayed(grid_search_fn)(timesarray, pulses)
        for timesarray in t_grid_stacked.T
    )
    return t_grid_stacked, loop_result


def grid_search_fom(
    t1_limits: tuple[float, float],
    t2_limits: tuple[float, float],
    t3_limits: tuple[float, float],
    counts: int | tuple[int, int, int],
    pulses: pd.DataFrame,
    cores: int = 4,
    use_chunks: bool = False,
) -> np.ndarray:
    t_grid_stacked, fom_values = search_grid(
        calculate_q_fom_critical_from_pulses,
        t1_limits,
        t2_limits,
        t3_limits,
        counts,
        pulses,
        cores,
        use_chunks
    )
    
    threshold_energies, threshold_search_starts = list(zip(*fom_values))
    threshold_energies = np.array(threshold_energies, dtype=float)
    threshold_search_starts = np.array(threshold_search_starts, dtype=float)

    return np.vstack(
        [t_grid_stacked, threshold_energies, threshold_search_starts],
        dtype=float
    ).T  # shape (n, 5)

In [ ]:
def calculate_plot_grid_dimensions(n: int, max_cols: int = 4) -> tuple[int, int]:
    ncols = min(n, max_cols)
    nrows = ceil(n / ncols)
    return (nrows, ncols)

In [ ]:
def display_plot_grid(
    grid_plot_fn: Callable[[mpl.axes.Axes, T], None],
    grid_plot_data: list[T],
    grid_count: int,
    max_cols: int
) -> tuple[mpl.figure.Figure, np.ndarray[mpl.axes.Axes]]:
    nrows, ncols = calculate_plot_grid_dimensions(grid_count, max_cols=max_cols)
    fig, axs = plt.subplots(
        nrows, ncols, figsize=(8*ncols, 8*nrows)
    )
    axs = axs.flatten()
    for ax, plot_data in zip(axs, grid_plot_data):
        grid_plot_fn(ax, plot_data)
    return fig, axs

In [ ]:
def pretty_format_duration(duration: float) -> str:
    out_seconds = duration % 60
    dur_minutes = int(duration / 60)
    if dur_minutes == 0:
        return f"{out_seconds:.1f} s"
    out_minutes = dur_minutes % 60
    dur_hours = int(dur_minutes / 60)
    if dur_hours == 0:
        return f"{out_minutes} m {out_seconds:.1f} s"
    else:
        return f"{dur_hours} h {out_minutes} m {out_seconds:.1f} s"

## Data Loading

### Loading Params

In [ ]:
experiment_ids = ["TB-26", "TB-LED_trigger"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

In [ ]:
# experiment_ids = input_experiment_ids()

In [ ]:
# # more here?
# experiment_neutron_data: ExperimentNeutronData = {
#     exp_id: {}
#     for exp_id in experiment_ids
# }

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )

# is_new_calibration = calib_input.lower() == "y"
# calibrated_energy_column: EnergyColumn = (
#     DetectorDataframeColumn.RECALIBRATED_ENERGY
#     if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
# )
# calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
# strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
# factory_fn = make_strategy_factory_fn(
#     strategy_factory, "nasa", False, settings)
# experiment_neutron_data = make_strategy_for_experiments(
#     experiment_neutron_data, factory_fn)

### Loading and Initial Processing

In [ ]:
figure_data = {k: {} for k in ["a", "b", "c", "d"]}

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Ensure that index matches between signals and CAEN data
for exp_id, exp_data in experiment_neutron_data.items():
    # print(exp_id)
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    # print(unclassified_df)
    signals_df = exp_data["signals_df"]
    # print(signals_df)

    unclassified_index: pd.Index = unclassified_df.index
    signals_index: pd.Index = signals_df.index
    clean_index = unclassified_index.intersection(signals_index)

    unclassified_df = unclassified_df.loc[clean_index]
    signals_df = signals_df.loc[clean_index]

    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df
    exp_data["signals_df"] = signals_df

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Data Processing

### Pulse Processing

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    signals_df = exp_data["signals_df"].copy().astype("int32")
    # print(signals_df.shape)

    flipped_raw_signals_df = correct_raw_signals(
        signals_df,
        use_max_adc=True
    )
    fixed_signals_df = correct_raw_signals(signals_df)
    print(flipped_raw_signals_df - fixed_signals_df)
    heights = fixed_signals_df.max(axis=1)
    # print(heights.max())
    flipped_raw_signals_df.columns = flipped_raw_signals_df.columns.map(int)
    fixed_signals_df.columns = fixed_signals_df.columns.map(int)

    psd_report["height"] = heights
    exp_data["no_baseline_signals"] = flipped_raw_signals_df
    exp_data["signals_df"] = fixed_signals_df
    exp_data[ExperimentDataKey.UNCLASSIFIED] = psd_report

In [ ]:
t1, t2, t3 = 95, 117, 345
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    signals_df = exp_data["signals_df"]
    no_base_signals_df = exp_data["no_baseline_signals"]

    integration_df = integrate_pulse_gates(signals_df, t1, t2, t3)
    no_base_integ_df = integrate_pulse_gates(no_base_signals_df, t1, t2, t3)

    integ_q_long = integration_df["Q_LONG"]
    integ_q_short = integration_df["Q_SHORT"]
    no_integ_q_long = no_base_integ_df["Q_LONG"]
    no_integ_q_short = no_base_integ_df["Q_SHORT"]
    print(integ_q_long.min(), integ_q_long.max())
    print(no_integ_q_long.min(), no_integ_q_long.max())

    integ_psd = (integ_q_long - integ_q_short) / integ_q_long
    no_integ_psd = (no_integ_q_long - no_integ_q_short) / no_integ_q_long

    psd_report["BASELINE_Q_LONG"] = integ_q_long
    psd_report["BASELINE_Q_SHORT"] = integ_q_short
    psd_report["BASELINE_PSD"] = integ_psd
    
    psd_report["NO_BASE_Q_LONG"] = no_integ_q_long
    psd_report["NO_BASE_Q_SHORT"] = no_integ_q_short
    psd_report["NO_BASE_PSD"] = no_integ_psd

### FOM Analysis - Base

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
adc_width = 20

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_adc_histogram(
        psd_report,
        # adc_col="height",
        adc_width=adc_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    helpers.stop()

### FOM Analysis - Manual/Baselined

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
adc_width = 1000

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_adc_histogram(
        psd_report,
        adc_col="BASELINE_Q_LONG",
        psd_col="BASELINE_PSD",
        adc_width=adc_width
    )
    exp_data["baseline_histo"] = Z
    exp_data["baseline_xe"] = xe
    exp_data["baseline_ye"] = ye
    exp_data["baseline_end_idx"] = min(end_scan_idx, len(Z))
    print(min(end_scan_idx, len(Z)))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data["baseline_histo"]
    xe = exp_data["baseline_xe"]
    ye = exp_data["baseline_ye"]
    end_scan_idx = exp_data["baseline_end_idx"]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data["baseline_slice_fits"] = df
        exp_data["baseline_bad_idxs"] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data["baseline_fom_results"] = df

if stop_here:
    helpers.stop()

### FOM Analysis - Manual/No Baseline

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
adc_width = 2000

for exp_id, exp_data in experiment_neutron_data.items():
    print(exp_id)
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_adc_histogram(
        psd_report,
        adc_col="NO_BASE_Q_LONG",
        psd_col="NO_BASE_PSD",
        adc_width=adc_width,
        psd_min=0,
        psd_max=1
    )
    exp_data["no_base_histo"] = Z
    exp_data["no_base_xe"] = xe
    exp_data["no_base_ye"] = ye
    exp_data["no_base_end_idx"] = min(end_scan_idx, len(Z))
    print(min(end_scan_idx, len(Z)))

In [ ]:
exp_id = "TB-26"
exp_data = experiment_neutron_data[exp_id]
Z = exp_data["no_base_histo"]
xe = exp_data["no_base_xe"]
ye = exp_data["no_base_ye"]

Z_disp = np.ma.masked_less(Z.T, 1)
# Z = exp_data["baseline_histo"]
# xe = exp_data["baseline_xe"]
# ye = exp_data["baseline_ye"]

# psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
# q_long = psd_report["NO_BASE_Q_LONG"].values
# psd = psd_report["NO_BASE_PSD"].values

# x_bin_min = 0
# x_bin_max = 620000
# x_bin_width = 2000
# x_bin_count = int((x_bin_max - x_bin_min) / x_bin_width)
# x_bins = np.linspace(x_bin_min, x_bin_max, x_bin_count+1)

# y_bin_min = 0
# y_bin_max = 0.5
# y_bin_count = 100
# y_bins = np.linspace(y_bin_min, y_bin_max, y_bin_count+1)

fig, ax = plt.subplots()
ax.pcolormesh(xe, ye, Z_disp, norm="log")
# *_, img = ax.hist2d(q_long, psd, bins=(x_bins, y_bins), cmin=1)
# ax.set_xlim(400000, None)
# ax.set_ylim(0.65, 0.95)

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data["no_base_histo"]
    xe = exp_data["no_base_xe"]
    ye = exp_data["no_base_ye"]
    end_scan_idx = exp_data["no_base_end_idx"]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data["no_base_slice_fits"] = df
        exp_data["no_base_bad_idxs"] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data["no_base_fom_results"] = df

if stop_here:
    helpers.stop()

### Figure 9a Processing

In [ ]:
plot_data = figure_data["a"]

In [ ]:
exp_trigger_type_map = {
    "TB-26": "CFD",
    "TB-leading_edge": "LED",
    "TB-LED_trigger": "LED"
}

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    trigger_plot_data = {}
    fom_df = exp_data[ExperimentDataKey.FOM_RESULTS]
    trigger_type = exp_trigger_type_map[exp_id]

    slice_e_min = fom_df["slice_energy_min"]
    slice_e_max = fom_df["slice_energy_max"]
    slice_fom = fom_df["fom"]
    
    slice_e_mid = (slice_e_min + slice_e_max) / 2

    trigger_plot_data["slice_e_mids"] = slice_e_mid
    trigger_plot_data["slice_foms"] = slice_fom
    plot_data[trigger_type] = trigger_plot_data

### Figure 9b Processing (plot already completed elsewhere)

In [ ]:
# # TODO try rewriting using Polars
# grid_side_lens = (1, 15, 15)
# t1_limits = (95.5, 95.5)
# t2_limits = (102, 130)
# t3_limits = (200, 380)

In [ ]:
# start_time = time.time()
# for exp_id, exp_data in experiment_neutron_data.items():
#     pulses = exp_data["signals_df"]
#     pulses = pulses.astype("uint32")
#     fom_search_results = grid_search_fom(
#         t1_limits,
#         t2_limits,
#         t3_limits,
#         grid_side_lens,
#         pulses
#     )
#     exp_data["fom_search_results"] = fom_search_results
# end_time = time.time()
# duration = end_time - start_time
# print(pretty_format_duration(duration))

In [ ]:
# for exp_id, exp_data in experiment_neutron_data.items():
#     fom_search_results = exp_data["fom_search_results"]
#     print(np.unique(fom_search_results[:, 0]))
#     print(np.unique(fom_search_results[:, 1]))
#     print(np.unique(fom_search_results[:, 2]))
#     *_, row_idx_min_fom, _ = np.nanargmin(fom_search_results, axis=0)
#     optimum = fom_search_results[row_idx_min_fom, :]
#     print(optimum)
#     # print(fom_search_results[:10, :])
#     exp_data["optimum"] = optimum

### Figure 9c Processing

In [ ]:
exp_id = "TB-26"
min_height = 5000
max_height = 6500

In [ ]:
exp_data = experiment_neutron_data[exp_id]
plot_data = figure_data["c"]

In [ ]:
signals_df = exp_data["no_baseline_signals"]

selected_signal = None
bad_signal = []

for signal_id, signal in signals_df.iterrows():
    if selected_signal is not None:
        break
    if signal_id in bad_signal:
        continue
    sig_height = signal.max()
    if not (min_height <= sig_height <= max_height):
        continue
    else:
        print(f"Signal ID = {signal_id}, height = {sig_height}")
        selected_signal = signal_id, signal

if selected_signal is None:
    raise Exception("No valid pulse found")
plot_data["selected_signal"] = selected_signal

### Figure 9d Processing

In [ ]:
exp_id = "TB-26"

In [ ]:
exp_data = experiment_neutron_data[exp_id]
plot_data = figure_data["d"]

In [ ]:
baseline_plot_data = {}
fom_df = exp_data["baseline_fom_results"]

slice_e_min = fom_df["slice_energy_min"]
slice_e_max = fom_df["slice_energy_max"]
slice_fom = fom_df["fom"]

slice_e_mid = (slice_e_min + slice_e_max) / 2

baseline_plot_data["slice_e_mids"] = slice_e_mid
baseline_plot_data["slice_foms"] = slice_fom
plot_data["baseline"] = baseline_plot_data

In [ ]:
no_base_plot_data = {}
fom_df = exp_data["no_base_fom_results"]

slice_e_min = fom_df["slice_energy_min"]
slice_e_max = fom_df["slice_energy_max"]
slice_fom = fom_df["fom"]

slice_e_mid = (slice_e_min + slice_e_max) / 2

no_base_plot_data["slice_e_mids"] = slice_e_mid
no_base_plot_data["slice_foms"] = slice_fom
plot_data["no_base"] = no_base_plot_data

## Plotting

### Plot Style Constants

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

### Plot Functions

In [ ]:
# for exp_name, data_dict in experiment_neutron_data.items():
#     print(exp_name)
#     psd_report = data_dict[ExperimentDataKey.UNCLASSIFIED]
#     fig, ax = plot_scatter(
#         psd_report[calibrated_energy_column.value],
#         psd_report[DetectorDataframeColumn.PSD.value]
#     )
#     ax.set_xlabel("Energy [MeVee]", fontsize=14)  # Update x-axis label
#     ax.set_ylabel("PSD", fontsize=14)
#     plt.show()

In [ ]:
def plot_figure_9a(
    ax: mpl.axes.Axes,
    x_zoom: tuple[float, float],
    y_zoom: tuple[float, float]
):
    fig_data = figure_data["a"]
    led_fig_data = fig_data["LED"]
    cfd_fig_data = fig_data["CFD"]

    global_params = {
        "lw": 3
    }
    led_plot_params = {
        "color": bg_bluegrey
    }
    cfd_plot_params = {
        "color": bg_blue
    }

    for sub_fig_data, sub_fig_params in [
        (led_fig_data, led_plot_params),
        (cfd_fig_data, cfd_plot_params)
    ]:
        e_mids = sub_fig_data["slice_e_mids"]
        foms = sub_fig_data["slice_foms"]
        params = {**global_params, **sub_fig_params}
        ax.plot(e_mids, foms, **params)

    ax.axhline(1.27, lw=2, linestyle="--", color=bg_grey)

    # ax.set_xlim(200, 300)
    # ax.set_ylim(1, 1.5)
    ax.set_xlim(*x_zoom)
    ax.set_ylim(*y_zoom)

    ax.set_xlabel("Total integral (ADC channel)", fontsize=fontsize)
    ax.set_ylabel("FOM", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)

In [ ]:
# TODO get 9b data, make 9b function

In [ ]:
def plot_figure_9c(ax: mpl.axes.Axes):
    fig_data = figure_data["c"]
    signal_id, signal = fig_data["selected_signal"]
    print(type(signal))

    signal_x = signal.index.map(int) * 2
    signal_y = signal.values

    # baseline_x = signal_x[:40]
    # baseline_y = signal_y[:40]

    ax.plot(signal_x, signal_y, "o-", color=bg_blue, ms=6, lw=3)
    # ax.plot(baseline_x, baseline_y, "o", ec=bg_blue, fc=None)
    ax.axvspan(0, 80, color=bg_grey, alpha=0.25)

    ax.set_xlabel("Time (ns)", fontsize=fontsize)
    ax.set_ylabel("Pulse height (ADC channels x1000)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")

In [ ]:
def plot_figure_9d(
    ax: mpl.axes.Axes,
    x_zoom: tuple[float, float],
    y_zoom: tuple[float, float]
):
    fig_data = figure_data["d"]
    baseline_plot_data = fig_data["baseline"]
    no_base_plot_data= fig_data["no_base"]
    
    global_params = {
        "lw": 3
    }
    baseline_plot_params = {
        "color": bg_blue
    }
    no_base_plot_params = {
        "color": bg_red
    }

    for sub_fig_data, sub_fig_params in [
        (baseline_plot_data, baseline_plot_params),
        (no_base_plot_data, no_base_plot_params)
    ]:
        e_mids = sub_fig_data["slice_e_mids"]
        foms = sub_fig_data["slice_foms"]
        params = {**global_params, **sub_fig_params}
        ax.plot(e_mids, foms, **params)

    ax.axhline(1.27, lw=2, linestyle="--", color=bg_grey)

    # ax.set_xlim(200, 300)
    # ax.set_ylim(1, 1.5)
    ax.set_xlim(*x_zoom)
    ax.set_ylim(*y_zoom)

    ax.set_xlabel("Total integral (ADC channel)", fontsize=fontsize)
    ax.set_ylabel("FOM", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)

In [ ]:
fig, ax = plt.subplots(
    layout="constrained"
)
plot_figure_9d(ax, (0, None), (0, None))

### Plot Creation

In [ ]:
fig, ax = plt.subplots(
    layout="constrained"
)
plot_figure_9a(ax, (0, 300), (0, 2))

In [ ]:
fig, ax = plt.subplots(
    layout="constrained"
)
plot_figure_9a(ax, (200, 300), (1, 1.5))

In [ ]:
fig, ax = plt.subplots(
    layout="constrained"
)
plot_figure_9c(ax)

## Done

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()